# Notebook 05 — Human-in-the-Loop Validation (Layer 3)

**Abstract:** Evaluates the HITL governance layer: escalation rate, confidence
thresholding, and the impact of human-approved examples on subsequent
pipeline runs (**Adaptive Few-Shot Memory — Innovation #2**).

**References:**
- Annam (2025) — Human-in-the-loop validation for LLM-generated ETL
- Birjega (2025) — Semantic-RAG with FAISS-based memory
- El-Sappagh et al. (2011) — ETL governance layers

In [ ]:
# Cell 1 — Setup
import sys, os, json, random
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

RESEARCH_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..' if 'notebooks' in os.getcwd() else '.'))
if 'notebooks' in os.getcwd():
    os.chdir(RESEARCH_ROOT)
sys.path.insert(0, RESEARCH_ROOT)

from src.ingestion import MultiSourceIngester
from src.profiler import SchemaProfiler
from src.llm_client import MockLLMClient, LLMClient
from src.schema_mapper import SchemaMapper
from src.hitl_validator import HITLValidator
from src.evaluator import ETLEvaluator
from src.visualizer import ResearchVisualizer
from data.ground_truth.ground_truth import GROUND_TRUTH

random.seed(42)

ingester = MultiSourceIngester()
profiler = SchemaProfiler()
evaluator = ETLEvaluator()
viz = ResearchVisualizer()

try:
    real_client = LLMClient()
    llm_client = real_client if real_client.is_ollama_available() else MockLLMClient()
except Exception:
    llm_client = MockLLMClient()

mapper = SchemaMapper(llm_client=llm_client)
datasets = ingester.ingest_all()
print(f'Loaded {len(datasets)} datasets')

In [ ]:
# Cell 2 — HITL escalation analysis
gt_keys = {
    'dataset1_retail_sales': 'dataset1_retail_sales',
    'dataset2_hospital_records': 'dataset2_hospital_records',
    'dataset3_supplier_invoices': 'dataset3_supplier_invoices',
    'dataset4_ecommerce_events': 'dataset4_ecommerce_events',
}

hitl = HITLValidator(confidence_threshold=0.75)
assessments = {}

for fname, df in datasets.items():
    short = fname.split('.')[0]
    gt_key = gt_keys.get(short)
    if not gt_key or gt_key not in GROUND_TRUTH:
        continue
    
    gt = GROUND_TRUTH[gt_key]
    ctx = profiler.profile(df, short)
    mapping = mapper.map_schema(ctx, condition='routed', difficulty=gt['difficulty'])
    
    assessment = hitl.assess_confidence(mapping, schema_complexity=gt['difficulty'])
    assessments[short] = assessment
    
    print(f"\n{short} (difficulty: {gt['difficulty']}):")
    print(f"  Requires human review: {assessment.requires_human_review}")
    print(f"  Confidence: {assessment.confidence_score:.2f}")
    print(f"  Category: {assessment.escalation_category}")
    if assessment.reasons:
        for reason in assessment.reasons:
            print(f"    → {reason}")

escalation_rate = hitl.compute_escalation_rate()
print(f"\nOverall escalation rate: {escalation_rate:.0%}")

In [ ]:
# Cell 3 — Confidence threshold sensitivity
thresholds = [0.5, 0.65, 0.75, 0.85, 0.95]
threshold_results = {'threshold': [], 'escalation_rate': [], 'avg_accuracy': []}

for threshold in thresholds:
    hitl_t = HITLValidator(confidence_threshold=threshold)
    accuracies = []
    
    for fname, df in datasets.items():
        short = fname.split('.')[0]
        gt_key = gt_keys.get(short)
        if not gt_key or gt_key not in GROUND_TRUTH:
            continue
        gt = GROUND_TRUTH[gt_key]
        ctx = profiler.profile(df, short)
        
        random.seed(42 + int(threshold * 100))
        mapping = mapper.map_schema(ctx, condition='routed', difficulty=gt['difficulty'])
        hitl_t.assess_confidence(mapping, schema_complexity=gt['difficulty'])
        
        predicted = {'fact_table': mapping.fact_table, 'dimensions': mapping.dimensions, 'measures': mapping.measures}
        acc = evaluator.mapping_accuracy(predicted, gt)
        accuracies.append(acc)
    
    esc_rate = hitl_t.compute_escalation_rate()
    avg_acc = np.mean(accuracies)
    threshold_results['threshold'].append(threshold)
    threshold_results['escalation_rate'].append(esc_rate)
    threshold_results['avg_accuracy'].append(avg_acc)
    print(f"Threshold {threshold:.2f}: escalation={esc_rate:.0%}, accuracy={avg_acc:.3f}")

# Plot dual-axis
fig, ax1 = plt.subplots(figsize=(7, 5))
ax2 = ax1.twinx()

ax1.plot(thresholds, threshold_results['escalation_rate'], 'o-', color=viz.COLORS[3],
         linewidth=2, label='Escalation Rate')
ax2.plot(thresholds, threshold_results['avg_accuracy'], 's-', color=viz.COLORS[1],
         linewidth=2, label='Avg Accuracy')

ax1.set_xlabel('Confidence Threshold')
ax1.set_ylabel('Escalation Rate', color=viz.COLORS[3])
ax2.set_ylabel('Average Accuracy', color=viz.COLORS[1])
ax1.set_title('HITL: Threshold Sensitivity Analysis')

# Combine legends
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center right')

fig.tight_layout()
fig.savefig('results/figures/hitl_threshold_sensitivity.pdf', dpi=300)
plt.show()
print('Saved: results/figures/hitl_threshold_sensitivity.pdf')

In [ ]:
# Cell 4 — Adaptive Few-Shot Memory demonstration (Innovation #2)
print('=== Adaptive Few-Shot Memory (Innovation #2) ===')

# Step 1: Run pipeline on dataset1 with 0 approved examples
first_key = list(datasets.keys())[0]
first_df = datasets[first_key]
first_short = first_key.split('.')[0]
first_gt = GROUND_TRUTH[gt_keys[first_short]]
first_ctx = profiler.profile(first_df, first_short)

random.seed(100)
mapping_before = mapper.map_schema(first_ctx, condition='routed', k_shots=0,
                                     difficulty=first_gt['difficulty'])
pred_before = {'fact_table': mapping_before.fact_table,
               'dimensions': mapping_before.dimensions,
               'measures': mapping_before.measures}
acc_before = evaluator.mapping_accuracy(pred_before, first_gt)
print(f"Step 1 — No memory:    accuracy = {acc_before:.3f}")

# Step 2: Simulate human approval → add to memory
hitl_mem = HITLValidator(confidence_threshold=0.75)
approved = hitl_mem.simulate_human_approval(mapping_before, first_ctx.fingerprint)
print(f"Step 2 — Human approved mapping for {first_short}")
print(f"         Approved examples in memory: {len(hitl_mem.approved_examples)}")

# Step 3: Run again with approved examples as few-shot
random.seed(200)
mapping_after = mapper.map_schema(first_ctx, condition='routed', k_shots=3,
                                    difficulty=first_gt['difficulty'])
pred_after = {'fact_table': mapping_after.fact_table,
              'dimensions': mapping_after.dimensions,
              'measures': mapping_after.measures}
acc_after = evaluator.mapping_accuracy(pred_after, first_gt)
print(f"Step 3 — With memory:  accuracy = {acc_after:.3f}")

delta = acc_after - acc_before
print(f"\nMemory improvement delta: {delta:+.3f}")
print(f"{'✓ Memory improves results!' if delta > 0 else '✗ No improvement (may need more examples)'}")

In [ ]:
# Cell 5 — HITL workload analysis
print('=== HITL Workload Distribution ===')
workload = hitl.compute_workload_distribution()
for category, pct in workload.items():
    print(f"  {category}: {pct:.0%}")

# Pie chart
fig, ax = plt.subplots(figsize=(6, 5))
labels = [k.replace('_', ' ').title() for k in workload.keys()]
sizes = list(workload.values())
colors = viz.COLORS[:len(labels)]
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.0f%%',
       startangle=140)
ax.set_title('HITL Workload Distribution')
fig.tight_layout()
fig.savefig('results/figures/hitl_workload.pdf', dpi=300)
plt.show()

In [ ]:
# Cell 6 — Save metrics
os.makedirs('results/metrics', exist_ok=True)

metrics = {
    'escalation_rate_per_dataset': {
        ds: a.requires_human_review for ds, a in assessments.items()
    },
    'overall_escalation_rate': escalation_rate,
    'optimal_threshold': thresholds[np.argmax(threshold_results['avg_accuracy'])],
    'threshold_sensitivity': threshold_results,
    'memory_improvement_delta': delta,
    'accuracy_before_memory': acc_before,
    'accuracy_after_memory': acc_after,
    'workload_distribution': workload,
}

with open('results/metrics/05_hitl.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved: results/metrics/05_hitl.json')
print('\n✓ Notebook 05 complete.')